# Stage 3 - Data Cleaning

Normalises the raw Google-Forms export into a database-standard, snake_case schema.

| Step | Action |
| --- | --- |
| 1 | Trim leading/trailing whitespace from column names |
| 2 | Drop corrupted/empty trailing rows (blank `Timestamp` **and** `Student Name`) |
| 3 | Rename all columns to database-standard snake_case |
| 4 | Consolidate the 6 course-title columns into `specialization` |
| 5 | Fill missing `upi_app` values with `'Not Applicable'` |

The transformation logic lives in `src/cleaner.py` so it can be unit-tested and re-run from a scheduler; this notebook demonstrates and exercises it.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "src" / "cleaner.py").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cleaner

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("project root :", PROJECT_ROOT)
print("raw source   :", cleaner.resolve_raw_data_path())

## 2. Load raw data and inspect the problems

The workbook stores the form labels with stray whitespace (e.g. `"Residential Background "`, `"Preferred Payment  "`) and appends two corrupted rows at the bottom.

In [ ]:
raw = cleaner.load_raw_data()
print(f"raw shape: {raw.shape[0]} rows x {raw.shape[1]} columns")

print("\ncolumns as stored (note the trailing whitespace):")
for column in raw.columns:
    print(f"  {column!r}")

In [ ]:
print("corrupted trailing rows (blank Timestamp AND Student Name):")
corrupted = raw[raw[["Timestamp", "Student Name"]].isna().all(axis=1)]
print(corrupted[["Timestamp", "Student Name", "Gender"]].to_string())

## 3. Clean

Order matters: names are stripped **before** the rename map is applied, otherwise the entries with trailing spaces in the source (`Residential Background`, `Part-time Work/Internship`, `Preferred Payment`) would silently fail to match.

In [ ]:
cleaned = cleaner.clean_data(raw)

print(f"cleaned shape: {cleaned.shape[0]} rows x {cleaned.shape[1]} columns")
print(f"rows dropped : {raw.shape[0] - cleaned.shape[0]}")
print("\ncleaned columns:")
for column in cleaned.columns:
    print(f"  {column}")

print("\nany column name still containing whitespace?")
print([c for c in cleaned.columns if str(c) != str(c).strip()] or "none")

In [ ]:
cleaned.head(10)

### Sanity checks

In [ ]:
assert list(cleaned.columns) == list(cleaner.CLEANED_COLUMNS), "unexpected column order"
assert not cleaned[["timestamp", "student_name"]].isna().any(axis=None), "corrupted rows survived"
assert cleaned["upi_app"].isna().sum() == 0, "upi_app still has nulls"
assert (cleaned["upi_app"] == "Not Applicable").sum() == 20, "unexpected upi_app fill count"
assert cleaned["specialization"].notna().all(), "every response should name a specialisation"
assert not any(column.startswith("Course Title") for column in cleaned.columns), "course columns not consolidated"

print("all checks passed")
print("rows with a filled upi_app:", (cleaned["upi_app"] == "Not Applicable").sum())

### `upi_app` after the fill

UPI is only asked when the preferred payment method is UPI, so a blank means the question did not apply - hence the explicit `'Not Applicable'` sentinel rather than `0`/empty.

In [ ]:
cleaned["upi_app"].value_counts(dropna=False)

### `specialization` after consolidation

Each respondent answered exactly one of the six course questions, so this is a straight coalesce; the join logic still handles multi-answer rows for future surveys.

In [ ]:
print("distinct specialisations:", cleaned["specialization"].nunique())
cleaned["specialization"].value_counts(dropna=False).head(20)

## 4. Export

Write the cleaned dataset to `data/processed/cleaned_student_budget.csv`, the input to Stage 4.

In [ ]:
destination = cleaner.export_cleaned_data(cleaned)
print("wrote:", destination)

roundtrip = pd.read_csv(destination)
print(f"verified on disk: {roundtrip.shape[0]} rows x {roundtrip.shape[1]} columns")

## Notes for Stage 4

Out of scope for this stage, but they will shape the modelling step:

1. **The money columns are ordinal range buckets, not numbers** - values look like `'< 1000 ₹'`, `'1000 - 2000 ₹'`, `'> 5000 ₹'`. Any arithmetic (totals, correlations) needs an explicit bucket-to-midpoint mapping first.
2. **Currency is baked into the string** (the rupee sign `₹`), so any numeric parse must strip it.
3. `preferred_payment` is only asked about a payment *method*; `upi_app` is conditional on it, which is why `'Not Applicable'` is the correct fill rather than a missing value.